In [3]:
import scipy
import numpy as np
import pandas as pd
import geopandas as gpd
from libpysal.weights import DistanceBand
from libpysal.weights.util import get_points_array
import libpysal
from spreg import Panel_RE_Lag
import os

# Compatibility Patch
if not hasattr(scipy, 'inf'):
    scipy.inf = np.inf

import math
from linearmodels.panel import RandomEffects
import statsmodels.api as sm
from pypandoc import convert_text

In [8]:
# ==============================================================================
# SDM MODEL - check for spatial lags
# ==============================================================================

# ==============================================================================
# 1. LOAD DATA
# ==============================================================================
excel_path = "Database for regression.xlsx" 
if not os.path.exists(excel_path):
    raise FileNotFoundError(f"File not found: {excel_path}")

# Load Excel - Fill explicit NaNs with 0 immediately
df_full = pd.read_excel(excel_path, sheet_name="Base de dados").fillna(0)

# Load Municipal Shapefile
shapefile_path = r"C:\Users\daves\OneDrive\Pessoal\Artigo\Economic History Review\Code\MT_Municipios_2022\MT_Municipios_2022.shp"
if not os.path.exists(shapefile_path):
     raise FileNotFoundError(f"Shapefile not found at: {shapefile_path}")
     
gdf_municipios = gpd.read_file(shapefile_path)

# Ensure CRS is projected
if gdf_municipios.crs.is_geographic:
    print("Converting CRS to projected coordinate system (SIRGAS 2000 / UTM zone 21S)...")
    gdf_municipios = gdf_municipios.to_crs(epsg=31981)

# ==============================================================================
# 2. BALANCE THE PANEL
# ==============================================================================
municipio_col_data = 'Município'

# Identify municipality column in shapefile
possible_cols = ['NM_MUN', 'NM_MUNICIP', 'NOME_MUN', 'Município', 'NOME']
municipio_col_shp = None
for col in possible_cols:
    if col in gdf_municipios.columns:
        municipio_col_shp = col
        break

if municipio_col_shp is None:
    municipio_col_shp = gdf_municipios.columns[0]

print(f"Using shapefile column: '{municipio_col_shp}'")

# Find municipalities common to shapefile and ALL years
years = sorted(df_full['Ano'].unique())
valid_municipios = set(gdf_municipios[municipio_col_shp].unique())

for year in years:
    municipios_in_year = set(df_full[df_full['Ano'] == year][municipio_col_data].unique())
    valid_municipios = valid_municipios.intersection(municipios_in_year)

valid_municipios = sorted(list(valid_municipios))

if len(valid_municipios) == 0:
    raise ValueError("No common municipalities found. Check spelling between Excel and Shapefile.")

# Filter to balanced panel
gdf_balanced = gdf_municipios[gdf_municipios[municipio_col_shp].isin(valid_municipios)].copy()
df_balanced = df_full[df_full[municipio_col_data].isin(valid_municipios)].copy()

# Sort Shapefile
gdf_balanced = gdf_balanced.sort_values(municipio_col_shp).reset_index(drop=True)
municipios_order = gdf_balanced[municipio_col_shp].tolist()

print(f"Balanced Panel: {len(municipios_order)} municipalities over {len(years)} years.")

# ==============================================================================
# 3. ROBUST VARIABLE TRANSFORMATION (Inline)
# ==============================================================================
vars_to_log = ['Production', 'Total Tillage', 'TFP', 'PIA', 'Annual Investment', 'Capital Stock']

print("\nApplying Robust Log Transformation...")

for v in vars_to_log:
    if v not in df_balanced.columns:
        print(f"Warning: Column '{v}' not found in Excel.")
        continue
        
    clean_name = f"l{v.replace(' ', '')}"
    
    # Inline logic replacing the custom function
    # 1. Fill NaNs with 0 and ensure float
    vals = df_balanced[v].fillna(0).values.astype(float)
    
    # 2. Create result array of zeros
    result = np.zeros_like(vals)
    
    # 3. Only calculate log where values are strictly positive
    mask_positive = vals > 0
    result[mask_positive] = np.log(vals[mask_positive])
    
    # Assign back to DataFrame
    df_balanced[clean_name] = result
    
    # Sanity check
    n_zeros = (df_balanced[clean_name] == 0).sum()
    print(f"   Created {clean_name}: {n_zeros} values set to 0 (originally <= 0 or NaN).")

# ==============================================================================
# 4. SPATIAL WEIGHTS (Inverse Distance)
# ==============================================================================
print("\nCreating Weights Matrix...")
gdf_balanced['centroid'] = gdf_balanced.geometry.centroid

# get_points_array helper logic inline (extract coordinates)
coords = np.column_stack((gdf_balanced['centroid'].x, gdf_balanced['centroid'].y))

# Threshold logic
distances = []
for i in range(len(coords)):
    dists = np.sqrt(((coords - coords[i])**2).sum(axis=1))
    dists = dists[dists > 0]
    if len(dists) > 0:
        distances.append(dists.min())

avg_nn = np.mean(distances)
threshold = avg_nn * 3.0  # Ensure connectivity

w = DistanceBand.from_array(coords, threshold=threshold, binary=False, alpha=-1.0, silence_warnings=True)
w.transform = 'r'

print(f"Weights Matrix Created. Neighbors (Mean): {w.mean_neighbors:.2f}")

# ==============================================================================
# 5. GENERATE SPATIAL LAGS (WX)
# ==============================================================================
x_vars = ['lTotalTillage', 'lTFP', 'lPIA', 'lAnnualInvestment', 'lCapitalStock']
wx_vars = []

# Ensure we use categorical sorting to match W exactly
df_balanced['MunicipioName_Cat'] = pd.Categorical(
    df_balanced[municipio_col_data], 
    categories=municipios_order, 
    ordered=True
)

# Sort strictly by Year, then Municipality
df_balanced = df_balanced.sort_values(by=['Ano', 'MunicipioName_Cat'])

df_list = []

print("\nCalculating Spatial Lags...")
for year in years:
    # Slice
    df_year = df_balanced[df_balanced['Ano'] == year].copy()
    
    # Ensure sort order matches W
    df_year = df_year.sort_values('MunicipioName_Cat').reset_index(drop=True)
    
    # Verify alignment
    if len(df_year) != w.n:
        raise ValueError(f"Year {year} mismatch. Data: {len(df_year)}, W: {w.n}")
    
    # Compute lags
    for var in x_vars:
        wx_name = f"W_{var}"
        # Direct calculation without try/except
        df_year[wx_name] = libpysal.weights.lag_spatial(w, df_year[var].values)
            
    df_list.append(df_year)

# Reassemble Panel
df_panel_final = pd.concat(df_list)

# Final Sort for SPREG: Municipality, then Year
df_panel_final = df_panel_final.sort_values(by=['MunicipioName_Cat', 'Ano']).reset_index(drop=True)

for var in x_vars:
    wx_vars.append(f"W_{var}")

# ==============================================================================
# 6. RUN MODEL WITH FINAL SAFETY CHECK
# ==============================================================================
y_var = df_panel_final[['lProduction']].values
x_sdm_vars = x_vars + wx_vars
x_mat = df_panel_final[x_sdm_vars].values

# Final robust check to replace any remaining -inf, inf, or nan with 0
y_var = np.nan_to_num(y_var, nan=0.0, posinf=0.0, neginf=0.0)
x_mat = np.nan_to_num(x_mat, nan=0.0, posinf=0.0, neginf=0.0)

print(f"\nRunning SDM...")
print(f"Y Shape: {y_var.shape}")
print(f"X Shape: {x_mat.shape}")

# Running model directly without try/except
model_sdm = Panel_RE_Lag(
    y_var, 
    x_mat, 
    w, 
    name_ds="Mato Grosso SDM Robust", 
    name_y="lProduction", 
    name_x=x_sdm_vars
)
print(model_sdm.summary)


# ==============================================================================
# GENERATE RTF TABLE
# ==============================================================================

filename = "SDM_Results.rtf"

# 1. Extract Data
# ------------------------------------------------------------------------------
# Coefficients
coeffs = [b[0] for b in model_sdm.betas]

# Standard Errors
se = list(model_sdm.std_err)

# P-values
p_values = [z[1] for z in model_sdm.z_stat]

# Variable Names
var_names = model_sdm.name_x
if len(coeffs) > len(var_names):
    var_names = ['Constant'] + var_names

# 2. Write RTF File
# ------------------------------------------------------------------------------
with open(filename, 'w') as f:
    # RTF Header
    f.write(r"{\rtf1\ansi\ansicpg1252\deff0\nouicompat\deflang1033{\fonttbl{\f0\fnil\fcharset0 Times New Roman;}}")
    f.write(r"{\*\generator Python Script}\viewkind4\uc1")
    f.write(r"\pard\sa200\sl276\slmult1\f0\fs24\lang9")

    # --- Row Definitions ---
    
    # 1. Header Row Definition (Top and Bottom Borders)
    header_row_def = r"\trowd \trqc\trgaph108" \
                     r"\clbrdrt\brdrs\brdrw10 \clbrdrb\brdrs\brdrw10 \cellx4500" \
                     r"\clbrdrt\brdrs\brdrw10 \clbrdrb\brdrs\brdrw10 \cellx7000"

    # 2. Body Row Definition (No Borders)
    body_row_def = r"\trowd \trqc\trgaph108" \
                   r"\cellx4500" \
                   r"\cellx7000"

    # 3. Bottom Row Definition (Bottom Border only - for the last stat)
    bottom_row_def = r"\trowd \trqc\trgaph108" \
                     r"\clbrdrb\brdrs\brdrw10 \cellx4500" \
                     r"\clbrdrb\brdrs\brdrw10 \cellx7000"

    # --- WRITE HEADER ---
    f.write(header_row_def)
    # CORRECTED HEADER: Shows lProduction as dependent variable
    f.write(r"\pard\intbl\widctlpar\qc\b Dependent variable\b0\cell \qc\b lProduction\b0\cell \row")

    # --- WRITE DATA ROWS ---
    for i in range(len(coeffs)):
        name = var_names[i]
        beta = coeffs[i]
        std = se[i]
        pval = p_values[i]

        # Stars 
        # * p < 0.05, ** p < 0.01, *** p < 0.001
        stars = ""
        if pval < 0.001:
            stars = "***"
        elif pval < 0.01:
            stars = "**"
        elif pval < 0.05:
            stars = "*"

        beta_str = f"{beta:.4f}{stars}"
        se_str = f"({std:.4f})"

        # Row 1: Variable Name & Coefficient
        f.write(body_row_def)
        f.write(rf"\pard\intbl\widctlpar\ql {name}\cell \qc {beta_str}\cell \row")

        # Row 2: SE (Empty left cell)
        f.write(body_row_def)
        f.write(rf"\pard\intbl\widctlpar\ql \cell \qc {se_str}\cell \row")

    # --- SUMMARY STATS ---
    
    # 1. Observations
    n_obs = f"{model_sdm.n}" if hasattr(model_sdm, 'n') else "N/A"
    f.write(body_row_def)
    f.write(rf"\pard\intbl\widctlpar\ql Observations\cell \qc {n_obs}\cell \row")

    # 2. Pseudo R2
    r2 = "N/A"
    if hasattr(model_sdm, 'pr2'):
        val = model_sdm.pr2.item() if hasattr(model_sdm.pr2, 'item') else model_sdm.pr2
        r2 = f"{val:.4f}"
    f.write(body_row_def)
    f.write(rf"\pard\intbl\widctlpar\ql Pseudo R2\cell \qc {r2}\cell \row")

    # 3. Log Likelihood (Last Row -> Use Bottom Border)
    ll = "N/A"
    if hasattr(model_sdm, 'logll'):
        val = model_sdm.logll.item() if hasattr(model_sdm.logll, 'item') else model_sdm.logll
        ll = f"{val:.4f}"
    
    # Apply the bottom border here to close the table
    f.write(bottom_row_def)
    f.write(rf"\pard\intbl\widctlpar\ql Log Likelihood\cell \qc {ll}\cell \row")

    f.write(r"}")

print(f"File '{filename}' generated successfully.")

Converting CRS to projected coordinate system (SIRGAS 2000 / UTM zone 21S)...
Using shapefile column: 'NM_MUN'
Balanced Panel: 138 municipalities over 5 years.

Applying Robust Log Transformation...
   Created lProduction: 314 values set to 0 (originally <= 0 or NaN).
   Created lTotalTillage: 298 values set to 0 (originally <= 0 or NaN).
   Created lTFP: 476 values set to 0 (originally <= 0 or NaN).
   Created lPIA: 337 values set to 0 (originally <= 0 or NaN).
   Created lAnnualInvestment: 297 values set to 0 (originally <= 0 or NaN).
   Created lCapitalStock: 416 values set to 0 (originally <= 0 or NaN).

Creating Weights Matrix...
Weights Matrix Created. Neighbors (Mean): 13.91

Calculating Spatial Lags...

Running SDM...
Y Shape: (690, 1)
X Shape: (690, 10)
REGRESSION
----------
SUMMARY OF OUTPUT: MAXIMUM LIKELIHOOD SPATIAL LAG PANEL - RANDOM EFFECTS
------------------------------------------------------------------------
Data set            :Mato Grosso SDM Robust
Weights matrix 

c:\Users\daves\anaconda3\Lib\site-packages\scipy\sparse\_data.py:128: RuntimeWarning: divide by zero encountered in power
  return self._with_data(data ** n)


In [6]:
# ==============================================================================
# GEOGRAPHICALLY WEIGHTS
# ==============================================================================
# Force pandas to display 4 decimals in the console
pd.options.display.float_format = '{:.4f}'.format
# ==============================================================================
# DATA PREPARATION (IMMEDIATE REGION)
# ==============================================================================
# Define the new grouping column
region_col = 'Região imediata'
# 1. Check if the column exists in the panel; if not, fetch from original DataFrame
if region_col not in df_panel_final.columns:
    print(f"Recovering '{region_col}' from original file...")
    # Check if it exists in df_full before attempting merge
    if region_col in df_full.columns:
        region_lookup = df_full[['Município', region_col]].drop_duplicates()
        df_panel_final = df_panel_final.merge(region_lookup, on='Município', how='left')
    else:
        raise KeyError(f"Column '{region_col}' was not found in the original Excel.")
# ==============================================================================
# DATA CLEANING AND VARIABLES (ERROR PREVENTION)
# ==============================================================================
x_vars_original = ['lTotalTillage', 'lTFP', 'lPIA', 'lAnnualInvestment', 'lCapitalStock']
wx_vars_selected = ['W_lTFP', 'W_lPIA'] 
final_x_vars = x_vars_original + wx_vars_selected
# Define only NUMERIC columns to clean
cols_to_clean = ['lProduction'] + final_x_vars
# Replace infinites and NaNs with 0 ONLY in numeric columns
df_panel_final[cols_to_clean] = df_panel_final[cols_to_clean].replace([np.inf, -np.inf], np.nan).fillna(0)
print(f"Data cleaned. Grouping by: {region_col}")
# ==============================================================================
# CALCULATE ELASTICITY BY IMMEDIATE REGION
# ==============================================================================
print(f"\n--- RUNNING MODELS BY: {region_col} ---")
# Get unique list of immediate regions
unique_regions = df_panel_final[region_col].dropna().unique()
regional_results = []
for region_name in unique_regions:
    # 1. Filter data only for this immediate region
    df_local = df_panel_final[df_panel_final[region_col] == region_name].copy()
    
    # 2. Check if there is sufficient data
    if len(df_local) < (len(final_x_vars) + 2):
        print(f"Skip {region_name}: Insufficient data ({len(df_local)} obs).")
        continue
    # 3. Set Index for Linearmodels (Panel)
    df_reg_local = df_local.set_index(['Município', 'Ano'])
    
    Y_local = df_reg_local['lProduction']
    X_local = df_reg_local[final_x_vars]
    X_local = sm.add_constant(X_local) 
    try:
        # Run Random Effects Panel
        mod_local = RandomEffects(Y_local, X_local)
        res_local = mod_local.fit()
        
        # 4. Store Results
        row = {
            'Immediate Region': region_name, 
            'R2_Overall': res_local.rsquared, 
            'Obs': res_local.nobs,
            'Cities': res_local.entity_info.total
        }
        
        # Save coefficients (Elasticities), p-values, and standard errors
        for var in final_x_vars:
            coef = res_local.params.get(var, 0)
            pval = res_local.pvalues.get(var, 1.0)
            stderr = res_local.std_errors.get(var, 0)
            
            # Add significance stars
            stars = ''
            if pval < 0.001:
                stars = '***'
            elif pval < 0.01:
                stars = '**'
            elif pval < 0.05:
                stars = '*'
            
            row[var] = f"{coef:.4f}{stars}"
            row[f'{var}_se'] = f"({stderr:.4f})"
            
        regional_results.append(row)
        
    except ZeroDivisionError:
        print(f"Pular {region_name}: Division by Zero Error (likely constant data).")
        continue
    except Exception as e:
        print(f"Erro ao calcular para {region_name}: {e}")
        continue
# ==============================================================================
# EXPORT RESULTS
# ==============================================================================
df_regional_elasticities = pd.DataFrame(regional_results)
if not df_regional_elasticities.empty:
    # Create a new DataFrame with alternating coefficient and standard error rows
    final_rows = []
    for idx, row in df_regional_elasticities.iterrows():
        # Coefficient row
        coef_row = {'Immediate Region': row['Immediate Region'], 'R2_Overall': row['R2_Overall'], 
                    'Obs': row['Obs'], 'Cities': row['Cities']}
        for var in final_x_vars:
            coef_row[var] = row[var]
        final_rows.append(coef_row)
        
        # Standard error row
        se_row = {'Immediate Region': '', 'R2_Overall': '', 'Obs': '', 'Cities': ''}
        for var in final_x_vars:
            se_row[var] = row[f'{var}_se']
        final_rows.append(se_row)
    
    df_sorted = pd.DataFrame(final_rows)
    
    print("\n=== RESULTS (TOP 10 IMMEDIATE REGIONS) ===")
    print(df_sorted.head(20))
    
    # Save to Excel with updated name
    filename_excel = "Elasticity_Regiao_Imediata.xlsx"
    df_sorted.to_excel(filename_excel, index=False)
    print(f"\nComplete table saved to: {filename_excel}")
    
else:
    print("No results calculated. Check if column 'Região imediata' exists and is populated.")

Data cleaned. Grouping by: Região imediata

--- RUNNING MODELS BY: Região imediata ---
Pular Diamantino: Division by Zero Error (likely constant data).
Pular Tangará da Serra: Division by Zero Error (likely constant data).
Pular Água Boa: Division by Zero Error (likely constant data).

=== RESULTS (TOP 10 IMMEDIATE REGIONS) ===
               Immediate Region R2_Overall Obs  Cities lTotalTillage  \
0                        Cuiabá     0.9111  65 13.0000     0.5747***   
1                                                           (0.1645)   
2                 Alta Floresta     0.8908  25  5.0000        0.6530   
3                                                           (0.8134)   
4                  Rondonópolis     0.9688  50 10.0000     0.4210***   
5                                                           (0.0808)   
6          Confresa - Vila Rica     0.9528  65 13.0000        0.1438   
7                                                           (0.1834)   
8               Barra 